# Eight town medians, no combined price

**Student Accommodation Finance | Matt Lenzie | Methods notebook | 2 November 2026**

The [H1 2026 report](https://studentaccommodationfinance.co.uk/insights/h1-2026-student-accommodation-finance-report/) publishes residential price medians for eight university postal towns. Can those medians, together with each town's observed sale count, tell us the median across all the underlying transactions?

**No: counts and group medians generally do not contain enough information to recover a pooled median.** This notebook makes the missing information visible. First it checks and reads the report's published display table. Then two explicitly invented, unitless arithmetic examples hold group counts and medians fixed while changing the pooled median. They are teaching examples, not property transactions or reconstructed research records.

The source study is **SAF-H1-2026**, covering activity from 1 January to 30 June 2026 with an evidence cutoff of 12 September 2026. This notebook adds an explanation and executable checks, not a new study or new observations. Its source file stays unchanged.

## How to use this notebook

Read the saved outputs without running anything, or run the code cells in order with Python 3. No packages, paid GPU, account token or Drive mount are required. The first code cell downloads one public CSV over HTTPS and verifies its expected hash. If that exact edition is unavailable or changes, the notebook stops rather than silently switching editions. Running in Colab may require a Google sign-in and is subject to its free runtime limits.


In [1]:
import csv
import hashlib
import io
import re
from statistics import median
from urllib.request import urlopen

SOURCE_URL = "https://studentaccommodationfinance.co.uk/reports/h1-2026/chart-data.csv"
EXPECTED_SHA256 = "a77ceb44b7aa14e9206576a1c1e43a81cbcbe69631e0163f31616d9e35cb98f8"
with urlopen(SOURCE_URL, timeout=30) as response:
    raw = response.read()
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256, "Source edition changed; inspect before using."
reader = csv.DictReader(io.StringIO(raw.decode("utf-8-sig")))
assert reader.fieldnames == ["exhibit", "title", "row", "measure", "value", "scope_note"]
rows = list(reader)
assert len(rows) == 64
print("Verified: frozen source SHA-256, six columns, 64 display observations.")
print("These are chart observations, not 64 underlying transactions.")


Verified: frozen source SHA-256, six columns, 64 display observations.
These are chart observations, not 64 underlying transactions.


## Keep the observed table separate from the demonstration

Exhibit 6 contains three measures for each town. We select only **Observed sales** and **Median price**; we do not substitute the terraced-property median. Each selected number retains its original scope note. These are general residential sales in exact postal-town fields, not verified student-property sales, university catchments or PBSA values per bed.

For the selection rules and registration lag, read the [original methodology](https://studentaccommodationfinance.co.uk/reports/h1-2026/methodology.md). For field-level exclusions and another checked import, use the [GitBook data reference](https://construction-capital.gitbook.io/student-accommodation-finance-data-reference/field-reference-and-exclusions).


In [2]:
EXPECTED_TOWNS = {"Oxford", "Bristol", "Exeter", "Southampton", "Manchester", "Leeds", "Nottingham", "Sheffield"}
selected = [r for r in rows if r["exhibit"] == "6" and r["measure"] in {"Observed sales", "Median price"}]
assert len(selected) == 16
indexed = {(r["row"], r["measure"]): r for r in selected}
assert len(indexed) == len(selected), "Duplicate town/measure."
assert {r["row"] for r in selected} == EXPECTED_TOWNS

def display_integer(value, currency=False):
    pattern = r"£(?:0|[1-9][0-9]{0,2}(?:,[0-9]{3})*|[1-9][0-9]*)" if currency else r"(?:0|[1-9][0-9]{0,2}(?:,[0-9]{3})*|[1-9][0-9]*)"
    assert re.fullmatch(pattern, value), "Unexpected display-value syntax."
    return int(value.replace("£", "").replace(",", ""))

town_table = []
for town in sorted(EXPECTED_TOWNS):
    count = indexed[town, "Observed sales"]
    price = indexed[town, "Median price"]
    assert count["scope_note"] == price["scope_note"] and price["scope_note"]
    town_table.append({"town": town, "observed_sales": display_integer(count["value"]),
                       "median_gbp": display_integer(price["value"], currency=True),
                       "scope_note": price["scope_note"]})
assert sum(r["observed_sales"] for r in town_table) == 19871
print(f"{'Postal town':<15} {'Observed sales':>15} {'Town median GBP':>18}")
for r in town_table:
    print(f"{r['town']:<15} {r['observed_sales']:>15,} {r['median_gbp']:>18,}")
print("\nObserved counts sum to 19,871. No pooled price median is calculated.")
print("Scope:", town_table[0]["scope_note"])


Postal town      Observed sales    Town median GBP
Bristol                   4,064            350,000
Exeter                      712            300,000
Leeds                     2,854            230,000
Manchester                4,053            235,000
Nottingham                3,358            220,000
Oxford                      559            440,000
Sheffield                 2,343            205,000
Southampton               1,928            295,000

Observed counts sum to 19,871. No pooled price median is calculated.
Scope: Observed H1 residential median price by postal town; not student HMO comparables or PBSA values per bed.


## Same summaries, different middle observations

A pooled median is found by ordering the pooled observations and taking the middle value, or the average of the two middle values for an even-sized sample. Town counts tell us how many observations exist. A town median tells us a central position within that town. Neither reveals all the observations that will occupy the middle after the towns are combined.

The next cell uses two **invented groups of three unitless numbers** in each of two worlds. In both worlds, group A has median 10 and group B has median 20. Even with the group counts known exactly, their pooled medians differ. These invented numbers are not appended to the report data or used to estimate actual prices.


In [3]:
# Teaching examples only: unitless invented numbers, not property records.
worlds = {
    "World 1": {"A": [1, 10, 100], "B": [10, 20, 21]},
    "World 2": {"A": [9, 10, 11], "B": [1, 20, 100]},
}

def summaries(groups):
    return {name: (len(values), median(values)) for name, values in groups.items()}

assert summaries(worlds["World 1"]) == summaries(worlds["World 2"]) == {"A": (3, 10), "B": (3, 20)}
pooled_medians = []
for name, groups in worlds.items():
    pooled = sorted(value for values in groups.values() for value in values)
    pooled_medians.append(median(pooled))
    weighted_group_median_average = sum(len(v) * median(v) for v in groups.values()) / len(pooled)
    print(name)
    print("  Group counts and medians:", summaries(groups))
    print("  Pooled ordered values:", pooled)
    print("  Actual median of these invented values:", median(pooled))
    print("  Count-weighted average of group medians:", weighted_group_median_average)
assert pooled_medians == [15.0, 10.5]
print("\nSame group summaries; different pooled medians. The summaries are insufficient.")


World 1
  Group counts and medians: {'A': (3, 10), 'B': (3, 20)}
  Pooled ordered values: [1, 10, 10, 20, 21, 100]
  Actual median of these invented values: 15.0
  Count-weighted average of group medians: 15.0
World 2
  Group counts and medians: {'A': (3, 10), 'B': (3, 20)}
  Pooled ordered values: [1, 9, 10, 11, 20, 100]
  Actual median of these invented values: 10.5
  Count-weighted average of group medians: 15.0

Same group summaries; different pooled medians. The summaries are insufficient.


## Why weighting does not fix the missing information

A count-weighted average of group medians is 15 in both invented worlds. It happens to equal the pooled median in World 1, but fails in World 2. Matching in one example does not establish a valid aggregation rule. The median of the two group medians also gives 15 and fails for the same reason.

Counts *can* be summed when the groups are disjoint and use the same inclusion rules. Means can be combined using appropriate counts when they are genuine, consistently defined arithmetic means. A median is a different statistic. Do not label an average or weighted average of the eight published town medians as the median of the 19,871 sales.

Recovering the actual pooled median would require sufficient information about the underlying ordered price distribution, usually the included transaction prices with consistent filtering, or an equivalent exact frequency distribution. The public display table does not supply that. This notebook deliberately leaves the pooled result uncomputed.

## A practical check before publishing a combined number

1. State the unit and population: general residential transactions in the selected exact postal towns.
2. Check whether the statistic is additive, a mean, a median or a ratio before choosing an aggregation rule.
3. Retain the activity window and source vintage. Registration lag is not solved by averaging across towns.
4. If the required inputs are missing, say that the combined statistic cannot be recovered from the available table.

The [Observable price comparison](https://observablehq.com/@matt-lenzie/university-town-sale-prices-two-views-of-h1-2026) is useful for inspecting the published town-level differences. The [research navigation guide](https://sites.google.com/view/saf-h1-2026-research-guide/home) connects the report, data, methods and tools. Those resources share the same evidence and authorship; they do not independently corroborate one another.

## Attribution and rights

By Matt Lenzie for Student Accommodation Finance, a commercial finance publication using research infrastructure shared with Construction Capital. AI assisted preparation and code checks. This notebook is a methods companion to SAF-H1-2026. Cite the [existing report DOI, 10.5281/zenodo.22733171](https://doi.org/10.5281/zenodo.22733171), and the relevant original sources when referring to its observations.

Original report, code and source-component rights remain reserved or subject to their respective owners' terms. No new general reuse or adaptation licence is granted here. Applicable statutory rights remain unaffected. This notebook is general research information, not a property valuation, lender offer or personal financial advice.
